# Demo suy luận Rich Gallery G1 — từ ảnh X-quang đến mặt nạ

**Ca trình diễn:** `IMG001598.jpeg` (nhãn cấp ảnh: tumor). Đây là ca test thật có kết quả tốt và có đầy đủ ba nguồn định vị.

Luồng demo: **ảnh + nhãn ảnh → 3 nguồn định vị → SAM proposal gallery → G1 score + upstream score → equal percentile-rank fusion → mask cuối**.

> Ranh giới khoa học: phần **Inference** không mở ground truth. Ground truth chỉ được đọc ở phần **Evaluation boundary** cuối notebook để tính Dice/IoU. Notebook replay đúng các output đã freeze của lần chạy cuối; không chọn lại ca hay ứng viên bằng ground truth.

## 0. Cấu hình và kiểm tra dữ liệu demo

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Markdown

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'project' / 'final_selector.py').is_file():
            return candidate
    raise FileNotFoundError('Không tìm thấy project/final_selector.py; hãy đặt REPO_ROOT thủ công.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CASE_ID = 'IMG001598.jpeg'
CASE_STEM = Path(CASE_ID).stem
CASE_ROOT = REPO_ROOT / 'outputs/analysis/round2_test_evidence_20260812/cases/three_source_complementarity'
GALLERY_NPZ = REPO_ROOT / f'outputs/private/round2_test_evidence_20260812/targeted/gallery/final_test_gallery/candidate_diagnostics/{CASE_STEM}.npz'
SCORE_NPZ = REPO_ROOT / 'outputs/private/round2_test_evidence_20260812/targeted/scores/final_test_scores/descriptor_evidence/0152_IMG001598.npz'

# Không liệt kê ground_truth_mask.png ở preflight inference để giữ ranh giới rõ ràng.
required_inference = [
    CASE_ROOT / 'xray.png',
    CASE_ROOT / 'layercam320_map.png',
    CASE_ROOT / 'classifier448_map.png',
    CASE_ROOT / 'external_saliency_map.png',
    GALLERY_NPZ, SCORE_NPZ,
]
missing = [str(path) for path in required_inference if not path.is_file()]
assert not missing, 'Thiếu artifact demo:\n' + '\n'.join(missing)
print(f'Repository : {REPO_ROOT}')
print(f'Case       : {CASE_ID}')
print('Inference artifacts: PASS')

## INFERENCE — không sử dụng ground truth

### 1. Đầu vào

Đầu vào online chỉ gồm ảnh X-quang và nhãn cấp ảnh `tumor=1`. Nhãn này cho biết **có tổn thương**, nhưng không cung cấp tọa độ, đường biên hay diện tích.

In [ ]:
xray = Image.open(CASE_ROOT / 'xray.png').convert('RGB')
display(Markdown(f'**{CASE_ID} — image-level label: tumor (1)**'))
display(xray.resize((310, 500)))

### 2. Ba nguồn bằng chứng định vị

- **LayerCAM-320:** bằng chứng không gian từ bộ phân loại nhị phân ở độ phân giải 320.
- **Classifier-448 / LayerCAM-448:** cùng nguyên lý nhưng ảnh lớn hơn để giữ tín hiệu tổn thương nhỏ.
- **BiomedCLIP saliency:** độ tương thích cục bộ giữa patch ảnh và ngữ nghĩa y sinh; bổ sung tín hiệu khác với gradient CAM.

Các bản đồ chỉ là bằng chứng mềm, chưa phải mặt nạ phân đoạn.

In [ ]:
def labeled_strip(items, width=260, label_height=34):
    prepared = []
    for label, image in items:
        im = image.convert('RGB')
        im.thumbnail((width, width))
        canvas = Image.new('RGB', (width, width + label_height), 'white')
        canvas.paste(im, ((width - im.width)//2, label_height))
        ImageDraw.Draw(canvas).text((8, 8), label, fill='black')
        prepared.append(canvas)
    strip = Image.new('RGB', (width * len(prepared), width + label_height), 'white')
    for i, im in enumerate(prepared):
        strip.paste(im, (i * width, 0))
    return strip

source_maps = [
    ('LayerCAM-320', Image.open(CASE_ROOT / 'layercam320_map.png')),
    ('LayerCAM-448', Image.open(CASE_ROOT / 'classifier448_map.png')),
    ('BiomedCLIP saliency', Image.open(CASE_ROOT / 'external_saliency_map.png')),
]
display(labeled_strip(source_maps))

### 3. Từ bản đồ mềm đến Rich Proposal Gallery

Các vùng nổi bật được chuyển thành point/box prompts. SAM ViT-B sinh nhiều proposal cho mỗi prompt. Sau gộp và loại trùng, ca này còn **144 ứng viên**. SAM score phản ánh chất lượng mask theo cơ chế nội tại của SAM; nó không trực tiếp khẳng định mask là khối u, vì vậy chưa được dùng một mình để quyết định.

In [ ]:
gallery = np.load(GALLERY_NPZ, allow_pickle=False)
masks = gallery['sam_masks'].astype(bool)
gallery_sources = gallery['proposal_source_ids'].astype(str)
assert masks.shape == (144, 320, 320)
source_counts = pd.Series(gallery_sources).value_counts().rename_axis('nguồn').reset_index(name='số ứng viên')
display(source_counts)

representatives = [
    ('LayerCAM-320 → SAM', Image.open(CASE_ROOT / 'layercam320_best_candidate.png')),
    ('LayerCAM-448 → SAM', Image.open(CASE_ROOT / 'classifier448_best_candidate.png')),
    ('BiomedCLIP → SAM', Image.open(CASE_ROOT / 'external_saliency_best_candidate.png')),
]
display(labeled_strip(representatives))

### 4. Chấm điểm từng ứng viên và hợp nhất thứ hạng

Với ứng viên $m_i$:

- **G1 logit** học từ đặc trưng RAD-DINO của vùng trong mask, vành ngữ cảnh cục bộ, độ tương phản và hình học;
- **upstream score** giữ bằng chứng sinh proposal từ CAM/saliency/SAM;
- hai thang điểm được đổi sang percentile rank trong cùng ảnh rồi hợp nhất:

$$s_i = 0.5\,R_{pct}(g_i) + 0.5\,R_{pct}(u_i).$$

Chọn $\arg\max_i s_i$; nếu hòa, ưu tiên G1 logit cao hơn rồi candidate index thấp hơn.

In [ ]:
from project.final_selector import average_percentile_rank, fixed_rank_fusion, select_candidate

scores = np.load(SCORE_NPZ, allow_pickle=False)
candidate_indices = scores['candidate_indices'].astype(int)
g1_logits = scores['candidate_logits'].astype(np.float64)
upstream_scores = scores['selection_scores'].astype(np.float64)
score_sources = scores['proposal_source_ids'].astype(str)
g1_rank = average_percentile_rank(g1_logits)
upstream_rank = average_percentile_rank(upstream_scores)
selected_index, fused_scores = select_candidate(g1_logits, upstream_scores)

assert np.allclose(fused_scores, 0.5 * g1_rank + 0.5 * upstream_rank)
assert selected_index == 87

top = np.argsort(-fused_scores, kind='stable')[:10]
ranking = pd.DataFrame({
    'hạng': np.arange(1, len(top)+1),
    'candidate': candidate_indices[top],
    'nguồn': score_sources[top],
    'G1 logit': g1_logits[top],
    'upstream': upstream_scores[top],
    'rank G1': g1_rank[top],
    'rank upstream': upstream_rank[top],
    'fusion': fused_scores[top],
})
display(ranking.style.format({c: '{:.4f}' for c in ['G1 logit','upstream','rank G1','rank upstream','fusion']}))
print(f'Kết quả selector: candidate #{selected_index}, source={score_sources[selected_index]}, fused={fused_scores[selected_index]:.6f}')

### 5. Mask đầu ra của pipeline (vẫn chưa đọc ground truth)

In [ ]:
selected_mask = masks[selected_index]
xray_320 = xray.resize((320, 320)).convert('RGB')
overlay = np.asarray(xray_320).copy()
red = np.zeros_like(overlay); red[..., 0] = 255
overlay[selected_mask] = (0.55 * overlay[selected_mask] + 0.45 * red[selected_mask]).astype(np.uint8)

display(labeled_strip([
    ('Ảnh đầu vào', xray_320),
    (f'Mask được chọn #{selected_index}', Image.fromarray((selected_mask * 255).astype(np.uint8))),
    ('Overlay dự đoán', Image.fromarray(overlay)),
]))
print('Inference hoàn tất. Chưa có ground truth nào được mở trong các cell phía trên.')

---
## EVALUATION BOUNDARY — từ đây mới được mở ground truth

Cell sau chỉ đánh giá mask đã được freeze. Nó không thay đổi proposal, score, thứ hạng hay lựa chọn.

In [ ]:
gt_path = CASE_ROOT / 'ground_truth_mask.png'
assert gt_path.is_file(), f'Thiếu ground truth đánh giá: {gt_path}'
gt_native = Image.open(gt_path).convert('L')
gt_320 = np.asarray(gt_native.resize((320, 320), Image.Resampling.NEAREST)) > 0

tp = int(np.logical_and(selected_mask, gt_320).sum())
fp = int(np.logical_and(selected_mask, ~gt_320).sum())
fn = int(np.logical_and(~selected_mask, gt_320).sum())
dice = 2 * tp / (2 * tp + fp + fn)
iou = tp / (tp + fp + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)

expected = json.loads((CASE_ROOT / 'metadata.json').read_text(encoding='utf-8'))
assert abs(dice - expected['selected_dice']) < 1e-12
assert abs(iou - expected['selected_iou']) < 1e-12

green = np.zeros_like(np.asarray(xray_320)); green[..., 1] = 255
gt_overlay = np.asarray(xray_320).copy()
gt_overlay[gt_320] = (0.55 * gt_overlay[gt_320] + 0.45 * green[gt_320]).astype(np.uint8)
display(labeled_strip([
    ('Dự đoán (đỏ)', Image.fromarray(overlay)),
    ('Ground truth (xanh)', Image.fromarray(gt_overlay)),
]))
display(pd.DataFrame([{
    'image_id': CASE_ID, 'candidate': selected_index, 'Dice': dice, 'IoU': iou,
    'Precision': precision, 'Recall': recall, 'Pred/GT area': selected_mask.sum()/gt_320.sum(),
}]).style.format({c: '{:.4f}' for c in ['Dice','IoU','Precision','Recall','Pred/GT area']}))

## Kết luận dùng khi thuyết trình

1. Pipeline chỉ nhận **ảnh và nhãn tumor cấp ảnh**; không có prompt người dùng hay mask trong inference.
2. Ba nguồn tạo bằng chứng không gian bổ sung, còn SAM biến bằng chứng mềm thành tập mask có biên rõ.
3. G1 trả lời “ứng viên này có giống tổn thương trong ngữ cảnh ảnh hay không”; upstream score giữ chất lượng và nguồn gốc proposal.
4. Percentile-rank fusion làm hai score khác thang đo có thể kết hợp ổn định.
5. Với ca đã khóa này, selector tự chọn candidate **87/144** và đạt **Dice 0,8575; IoU 0,7505**. Ground truth chỉ xuất hiện sau khi lựa chọn đã hoàn tất.

**Lưu ý khi demo:** đây là một ca thành công đại diện để giải thích cơ chế, không phải bằng chứng thay thế cho kết quả trung bình toàn bộ test set.